# ML-08 — Model vs Week-4 Baseline

**Lane:** Refresh / Content Opportunity Scoring

This notebook tests whether a simple ML ranking model improves the Week-4 hand-written baseline on a future-window decline outcome. The evaluation uses the same March 2026 decision slice and the same Precision@50 metric as the baseline, while the outcome is measured in April so it is genuinely future-looking.

> Run this notebook top-to-bottom in Colab. It requires the FlyRank Hugging Face dataset and the `HF_TOKEN` Colab Secret.

## 1. Method choice and why

I chose a **Random Forest classifier** because the lane is a ranking/scoring problem with a small set of numeric search-performance signals. A forest can capture nonlinear relationships and interactions without requiring me to hand-specify them. I will not treat complexity as success: the model only earns its place if its validated Precision@50 is better than the transparent Week-4 baseline.

The prediction target is a future outcome: a page is labelled declining when next month's impressions are at least 20% lower than the current month's impressions. This is a practical proxy for future search visibility decline, not proof of a causal refresh opportunity.

Features are limited to information available at the March decision moment: March impressions, clicks, CTR, average position, and GA4-data availability. IDs are used only for grouping/reporting, never as features.


In [ ]:
%pip -q install duckdb scikit-learn pandas pyarrow
import duckdb, os, numpy as np, pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier

con = duckdb.connect()
token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    token = token or userdata.get('HF_TOKEN')
except Exception:
    pass
if not token:
    raise RuntimeError('HF_TOKEN is missing. Add it to Colab Secrets and enable Notebook access.')
safe_token = token.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

def rel(month):
    return f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={month}/*.parquet')"

print('DuckDB + Hugging Face connection ready.')


## 2. Split design

I use a **time-aware split**. January and February 2026 are training periods because their next-month outcomes (February and March) are observable. March 2026 is the held-out test decision month, and its label is based on April 2026. This prevents the model from learning from information that would only exist after the March decision.

The Week-4 baseline is evaluated on exactly the same March test rows and April outcome. This makes the model-vs-baseline comparison apples-to-apples.


In [ ]:
MONTHS = ['2026-01','2026-02','2026-03','2026-04']

monthly_parts = []
for m in MONTHS:
    q = f"""
    SELECT client_hash_id AS client_id, content_hash_id AS content_id, month,
           SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position,
           BOOL_OR(gsc_data_available IS TRUE) AS gsc_available,
           BOOL_OR(ga4_data_available IS TRUE) AS ga4_available
    FROM {rel(m)}
    GROUP BY 1,2,3
    """
    monthly_parts.append(con.sql(q).df())
monthly = pd.concat(monthly_parts, ignore_index=True)
monthly['ctr'] = np.where(monthly['impressions'] > 0, monthly['clicks'] / monthly['impressions'], 0.0)
monthly = monthly.sort_values(['client_id','content_id','month'])
monthly['next_month_impressions'] = monthly.groupby(['client_id','content_id'])['impressions'].shift(-1)
monthly['is_declining_future'] = (monthly['next_month_impressions'] < 0.8 * monthly['impressions']).astype(int)
monthly.loc[monthly['next_month_impressions'].isna(), 'is_declining_future'] = np.nan
train = monthly[monthly['month'].isin(['2026-01','2026-02'])].dropna(subset=['is_declining_future']).copy()
test = monthly[monthly['month'].eq('2026-03')].dropna(subset=['is_declining_future']).copy()
print(f'Train rows: {len(train):,} | Test rows: {len(test):,}')
print(f'Test future-decline rate: {test.is_declining_future.mean():.3f}')
display(test[['month','client_id','content_id','impressions','clicks','ctr','avg_position','is_declining_future']].head())


## 3. Train + compare vs my baseline

The primary metric is **Precision@50** because the operational decision is the quality of the first 50 pages in the review queue. The baseline is frozen from Week 4: score by March impressions only when impressions are at least 300 and average position is between 4 and 20; otherwise score 0. The model is ranked by predicted probability of the future-decline label.


In [ ]:
FEATURES = ['impressions','clicks','ctr','avg_position','ga4_available']
X_train = train[FEATURES].copy(); X_test = test[FEATURES].copy(); y_train = train['is_declining_future'].astype(int); y_test = test['is_declining_future'].astype(int)
for frame in [X_train, X_test]:
    frame['ga4_available'] = frame['ga4_available'].fillna(False).astype(int)
    frame['avg_position'] = frame['avg_position'].replace([np.inf,-np.inf],np.nan).fillna(999.0)
    for c in ['impressions','clicks','ctr']:
        frame[c] = frame[c].replace([np.inf,-np.inf],np.nan).fillna(0.0)
model = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20, random_state=42, class_weight='balanced_subsample', n_jobs=-1)
model.fit(X_train, y_train)
test['model_score'] = model.predict_proba(X_test)[:,1]
test['baseline_score'] = np.where((test['impressions'] >= 300) & (test['avg_position'].between(4,20)), test['impressions'], 0.0)
def precision_at_k(frame, score_col, k=50):
    top = frame.sort_values(score_col, ascending=False).head(k)
    return float(top['is_declining_future'].mean()) if len(top) else np.nan
K=50
results = pd.DataFrame({'method':['Week-4 baseline','Random Forest'], 'precision_at_50':[precision_at_k(test,'baseline_score',K),precision_at_k(test,'model_score',K)], 'n_test':[len(test),len(test)]})
display(results)
print(f'Baseline Precision@50: {results.loc[0,"precision_at_50"]:.3f}')
print(f'Model Precision@50: {results.loc[1,"precision_at_50"]:.3f}')


## 4. Errors and interpretation

A useful error review should inspect false positives in the model's top queue and compare them with baseline picks. A high model score is not proof that a refresh is correct: the future label is only an observed traffic/search proxy and can be affected by seasonality, tracking changes, demand changes, or other factors.


In [ ]:
top_model = test.sort_values('model_score', ascending=False).head(50).copy()
top_model['error'] = np.where(top_model['is_declining_future'].eq(0), 'false_positive', 'correct_positive')
print('Top-50 model queue error mix:')
display(top_model['error'].value_counts().rename_axis('error_type').to_frame('n'))
print('Top model false positives:')
display(top_model[top_model['error'].eq('false_positive')][['client_id','content_id','impressions','ctr','avg_position','model_score']].head(10))
importance = pd.DataFrame({'feature':FEATURES,'importance':model.feature_importances_}).sort_values('importance',ascending=False)
print('Random Forest feature importance (direction is not causal):')
display(importance)
baseline_top=test.sort_values('baseline_score',ascending=False).head(50)
print(f'Baseline top-50 positive count: {baseline_top.is_declining_future.sum():.0f}')
print(f'Model top-50 positive count: {top_model.is_declining_future.sum():.0f}')


## 5. Interpretation

The model should be retained only if its held-out Precision@50 is meaningfully better than the Week-4 baseline on the same March test slice. If the improvement is small, unstable, or hard to explain, the transparent baseline may be the better operational choice.

Feature importance is descriptive, not causal. The future-decline label is a proxy, so errors do not automatically mean that a page should or should not be refreshed. Human review remains the final decision gate.


## Self-check

- [x] Method choice is explained and tied to the lane.
- [x] Time-aware train/test design is explicit.
- [x] Test period is March 2026; outcome uses April 2026.
- [x] Model and frozen Week-4 baseline use the same test rows and Precision@50.
- [x] No IDs are model features.
- [x] No current/future outcome fields are used as prediction features.
- [x] Errors and feature interpretation are included.
- [ ] Run **Runtime → Run all** in Colab and save the executed notebook back to GitHub before submitting.
